In [ ]:
import picsure

In [ ]:
open_hpds_session = picsure.connect(
    picsure.Platform.BDC_OPEN,
    dev_mode=True
)

In [ ]:
facets = open_hpds_session.facets()
facets.add("dataset_id", "phs000810")
facets.add("dataset_id", "phs000007")
facets.view()

In [ ]:
results = open_hpds_session.searchDictionary("age", facets=facets)
results

In [ ]:
# phs000007 - Will be used to replace the phs000810 clause
age5_phs000007 = results[results["display"] == "age5"]
age5_phs000007 = age5_phs000007[age5_phs000007["name"] == "phv00177938"]
age5_phs000007_clause = picsure.buildClause(
    age5_phs000007["conceptPath"].iloc[0],
    picsure.PhenotypicFilterType.FILTER,
    min=30,
    max=40
)

# phs000810 - Will be used to replace the phs000007 clause
age_immi_phs000810 = results[results["display"] == "AGE_IMMI"]
age_immi_phs000810_clause = picsure.buildClause(
    age_immi_phs000810["conceptPath"].iloc[0],
    picsure.PhenotypicFilterType.FILTER,
    min=30,
    max=40
)

In [ ]:
fhs_facet = open_hpds_session.facets()
fhs_facet.add("dataset_id", "phs000007")
fhs_sex_results = open_hpds_session.searchDictionary("phv00253990", facets=fhs_facet)
fhs_sex_results

In [ ]:
fhs_sex_male_clause = picsure.buildClause(
    fhs_sex_results['conceptPath'].iloc[0],
    picsure.PhenotypicFilterType.FILTER,
    ["Male"]
)

fhsMaleAnd30To40 = picsure.buildClauseGroup(
    [fhs_sex_male_clause, age5_phs000007_clause],
    operator=picsure.GroupOperator.AND
)

fhs_sex_female_clause = picsure.buildClause(
    fhs_sex_results['conceptPath'].iloc[0],
    picsure.PhenotypicFilterType.FILTER,
    ["Female"]
)

fhsFemaleAnd30To40 = picsure.buildClauseGroup(
    [fhs_sex_female_clause, age5_phs000007_clause],
    operator=picsure.GroupOperator.AND
)

fhs_FemaleAges30to40_OR_MaleAges30to40 = picsure.buildClauseGroup(
    [fhsFemaleAnd30To40, fhsMaleAnd30To40],
    operator=picsure.GroupOperator.OR
)

fhs_FemaleAges30to40_OR_MaleAges30to40.to_query_json()

In [ ]:
open_hpds_session.runQuery(picsure.buildQuery(phenotypicFilter=fhs_FemaleAges30to40_OR_MaleAges30to40))
# Results are within expected margin. Verified results with UI. UI Result: 77±3

In [ ]:
# We will now replace all of the age5_phs000007_clause with age_immi_phs000810_clause
fhs_FemaleAges30to40_OR_MaleAges30to40 = picsure.replaceClause(
    fhs_FemaleAges30to40_OR_MaleAges30to40,
    age5_phs000007_clause,
    age_immi_phs000810_clause
)

fhs_FemaleAges30to40_OR_MaleAges30to40.to_query_json()

In [ ]:
open_hpds_session.runQuery(picsure.buildQuery(phenotypicFilter=fhs_FemaleAges30to40_OR_MaleAges30to40))

In [ ]:
# We can also remove sub-queries from a query
fhs_Female_OR_Male = picsure.removeSubQuery(
    fhs_FemaleAges30to40_OR_MaleAges30to40,
    age_immi_phs000810_clause
)

fhs_Female_OR_Male.to_query_json()

In [ ]:
open_hpds_session.runQuery(picsure.buildQuery(phenotypicFilter=fhs_Female_OR_Male))